# Mount Drive and Load Dataset

In [1]:
from google.colab import drive
import math
import numpy as np

drive.mount('/content/drive')

data_path = "/content/drive/MyDrive/world_model_dataset/dataset.npz"
data = np.load(data_path)

observations = data["observations"]  # shape: (N, 64, 64, 3)
actions = data["actions"]            # shape: (N, 3)
rewards = data["rewards"]            # shape: (N,)
dones = data["dones"]                # shape: (N,)

Mounted at /content/drive


In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


# VAE

In [3]:
import torch
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, latent_dim=512):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # 64x64 -> 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 32x32 -> 16x16
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 16x16 -> 8x8
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),# 8x8 -> 4x4
            nn.ReLU()
        )

        # Latent space (ONLY mu is used)
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256*4*4, latent_dim)

        # Decoder fully connected
        self.fc_dec = nn.Linear(latent_dim, 256*4*4)

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 4x4 -> 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),    # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),     # 32x32 -> 64x64
            nn.Sigmoid()  # output in [0,1]
        )

    def encode(self, x):

        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        return mu

    def forward(self, x):
        return self.encode(x)

# Load VAE

In [4]:

latent_dim = 512
vae = VAE(latent_dim=latent_dim)
vae.load_state_dict(torch.load("vae_carracing.pth", map_location=device))
vae.to(device)
vae.eval()


for p in vae.parameters():
    p.requires_grad = False

# Convert images → latents

In [5]:
latents = []

vae.eval()

with torch.no_grad():
    for i in range(len(observations)):
        img = torch.tensor(observations[i]).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        img = img.to(device)
        z = vae.encode(img)  # returns single tensor
        latents.append(z.cpu().numpy())

latents = np.concatenate(latents, axis=0)

# MDN Loss

In [ ]:
# def mdn_loss(pi, mu, sigma, target):
#     # target: [B, T, latent_dim]
#     target = target.unsqueeze(-1)  # [B, T, latent_dim, 1]

#     exponent = -0.5 * ((target - mu) / sigma) ** 2
#     exponent = exponent.sum(dim=2)  # sum over latent_dim

#     latent_dim = target.size(2)
#     log_coef = -torch.log(sigma).sum(dim=2) - 0.5 * latent_dim * np.log(2 * np.pi)

#     log_probs = log_coef + exponent
#     weighted_log_probs = torch.log(pi + 1e-8) + log_probs

#     max_log, _ = weighted_log_probs.max(dim=-1, keepdim=True)
#     log_sum = max_log.squeeze(-1) + torch.log(
#         torch.sum(torch.exp(weighted_log_probs - max_log), dim=-1)
#     )

#     return -log_sum.mean()

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# World Model

In [13]:

import torch.nn.functional as F

class WorldModelTransformerMDN(nn.Module):
    def __init__(self, latent_dim, action_dim, d_model=256, nhead=8, num_layers=4, num_mixtures=5):
        super().__init__()

        self.num_mixtures = num_mixtures

        self.input_proj = nn.Linear(latent_dim + action_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # MDN heads
        self.fc_mu = nn.Linear(d_model, latent_dim * num_mixtures)
        self.fc_sigma = nn.Linear(d_model, latent_dim * num_mixtures)
        self.fc_pi = nn.Linear(d_model, num_mixtures)

        # ✅ Reward head
        self.fc_reward = nn.Linear(d_model, 1)

    def forward(self, z, a):
        x = torch.cat([z, a], dim=-1)
        x = self.input_proj(x)
        x = self.pos_enc(x)
        h = self.transformer(x)

        batch_size, seq_len, _ = h.shape

        mu = self.fc_mu(h).view(batch_size, seq_len, -1, self.num_mixtures)
        sigma = torch.exp(self.fc_sigma(h)).view(batch_size, seq_len, -1, self.num_mixtures)
        pi = F.softmax(self.fc_pi(h), dim=-1)

        reward_pred = self.fc_reward(h).squeeze(-1)  # [batch, seq_len]

        return mu, sigma, pi, reward_pred


In [20]:
model = WorldModelTransformerMDN(latent_dim=512, action_dim=3).to(device)
checkpoint = torch.load("/content/world_model_mdn_checkpoint (1).pth")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


WorldModelTransformerMDN(
  (input_proj): Linear(in_features=515, out_features=256, bias=True)
  (pos_enc): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc_mu): Linear(in_features=256, out_features=2560, bias=True)
  (fc_sigma): Linear(in_features=256, out_features=2560, bias=True)
  (fc_pi): Linear(in_features

# Controller

In [24]:
import torch
import torch.nn as nn

class LinearController(nn.Module):
    def __init__(self, latent_dim, action_dim):
        super().__init__()
        self.linear = nn.Linear(latent_dim, action_dim)

    def forward(self, z):
        raw_actions = self.linear(z)
        steering = torch.tanh(raw_actions[:, 0:1])
        gas = torch.sigmoid(raw_actions[:, 1:2])
        brake = torch.sigmoid(raw_actions[:, 2:3]) * 0.8
        brake = brake * (1 - gas)
        return torch.cat([steering, gas, brake], dim=1)

# Rollout in World Model (Cumulative Reward)

In [40]:
from collections import deque
def rollout(controller, z_init, horizon=1200, context_len=16):
    z = z_init.unsqueeze(0).to(device)  # [1, latent_dim]
    cumulative_reward = 0.0

    # history deques to store past latent states and actions
    history_z = deque(maxlen=context_len)
    history_a = deque(maxlen=context_len)

    for t in range(horizon):
        a = controller(z)

        # append current state and action to history
        history_z.append(z.squeeze(0))  # shape [latent_dim]
        history_a.append(a.squeeze(0))  # shape [action_dim]

        # stack history into sequences for transformer input
        z_seq = torch.stack(tuple(history_z), dim=0).unsqueeze(0)  # [1, seq_len, latent_dim]
        a_seq = torch.stack(tuple(history_a), dim=0).unsqueeze(0)  # [1, seq_len, action_dim]

        with torch.no_grad():
            mu, sigma, pi, r_pred = model(z_seq, a_seq)

            # pick mean of first mixture as deterministic prediction
            z = mu[:, -1, :, 0]  # last timestep in sequence
            cumulative_reward += r_pred[:, -1].item()

    return cumulative_reward

# CMA-ES Controller Training

In [72]:
!pip install cma
import cma

latent_dim = 512
action_dim = 3
controller = LinearController(latent_dim, action_dim).to(device)

params = controller.state_dict()['linear.weight'].flatten().cpu().numpy()
es = cma.CMAEvolutionStrategy(params, 0.5)  # sigma=0.5

best_reward = -np.inf
num_starts = 1
num_generations = 3

for generation in range(num_generations):
    solutions = es.ask()
    rewards = []
    z_inits = [torch.tensor(latents[i], dtype=torch.float32).to(device) for i in range(num_starts)]


    for sol in solutions:
        weight = torch.tensor(sol.reshape(action_dim, latent_dim), dtype=torch.float32).to(device)
        controller.linear.weight.data = weight
        controller.linear.bias.data.zero_()  # optional

        rewards_per_sol = []
        for z_init in z_inits:
            r = rollout(controller, z_init)
            rewards_per_sol.append(r)

        rewards.append(-np.mean(rewards_per_sol))

    es.tell(solutions, rewards)

    best_reward = max(best_reward, -min(rewards))
    print(f"Generation {generation+1} | Best reward so far: {best_reward:.4f}")


(13_w,26)-aCMA-ES (mu_w=7.6,w_1=23%) in dimension 1536 (seed=132100, Mon Mar  9 16:33:44 2026)
Generation 1 | Best reward so far: 893.9116
Generation 2 | Best reward so far: 908.9913
Generation 3 | Best reward so far: 953.6809


# Using Controller in Real Environment

In [73]:
best_solution = es.result.xbest
print("Best reward found:", -es.result.fbest)

Best reward found: 953.6809220537543


In [74]:
best_weight = torch.tensor(
    best_solution.reshape(action_dim, latent_dim),
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    controller.linear.weight.copy_(best_weight)
    controller.linear.bias.zero_()

In [75]:
torch.save({
    "controller_state_dict": controller.state_dict(),
    "latent_dim": latent_dim,
    "action_dim": action_dim
}, "controller_best.pth")

In [76]:
checkpoint = torch.load("controller_best.pth", map_location=device)

controller = LinearController(
    checkpoint["latent_dim"],
    checkpoint["action_dim"]
).to(device)

controller.load_state_dict(checkpoint["controller_state_dict"])
controller.eval()

LinearController(
  (linear): Linear(in_features=512, out_features=3, bias=True)
)

In [78]:
from collections import deque
import torch
import cv2
import numpy as np
import gymnasium as gym

# ---- setup ----
context_len = 16  # same as training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

history_z = deque(maxlen=context_len)
history_a = deque(maxlen=context_len)

env = gym.make("CarRacing-v3", render_mode="human")
obs, _ = env.reset()
done = False
total_reward = 0.0

while not done:
    # 1️⃣ Preprocess image
    sm_img = cv2.resize(obs, (64, 64))
    img = torch.tensor(sm_img).permute(2,0,1).float()/255.0
    img = img.unsqueeze(0).to(device)

    # 2️⃣ Encode to latent
    with torch.no_grad():
        z = vae.encode(img)  # shape [1, latent_dim]

    # 3️⃣ Maintain history for transformer context
    history_z.append(z.squeeze(0))

    # pad history if we don’t have enough
    while len(history_z) < context_len:
        history_z.append(torch.zeros_like(z.squeeze(0)))

    z_seq = torch.stack(tuple(history_z), dim=0).unsqueeze(0)  # [1, seq_len, latent_dim]

    # 4️⃣ Controller predicts action based on last latent in sequence
    with torch.no_grad():
        a = controller(z_seq[:, -1, :])  # only last latent to controller
        history_a.append(a.squeeze(0))
        while len(history_a) < context_len:
            history_a.append(torch.zeros_like(a.squeeze(0)))

    # 5️⃣ Step environment
    obs, reward, terminated, truncated, info = env.step(a.cpu().numpy()[0])
    total_reward += reward
    print(f"Cumulative reward: {total_reward:.3f}")

    done = terminated or truncated

print("Final Total Reward:", total_reward)
env.close()

Cumulative reward: 7.120
Cumulative reward: 7.020
Cumulative reward: 6.920
Cumulative reward: 6.820
Cumulative reward: 6.720
Cumulative reward: 6.620
Cumulative reward: 6.520
Cumulative reward: 6.420
Cumulative reward: 6.320
Cumulative reward: 6.220
Cumulative reward: 6.120
Cumulative reward: 6.020
Cumulative reward: 5.920
Cumulative reward: 5.820
Cumulative reward: 5.720
Cumulative reward: 5.620
Cumulative reward: 5.520
Cumulative reward: 5.420
Cumulative reward: 5.320
Cumulative reward: 5.220
Cumulative reward: 5.120
Cumulative reward: 8.630
Cumulative reward: 8.530
Cumulative reward: 8.430
Cumulative reward: 8.330
Cumulative reward: 8.230
Cumulative reward: 8.130
Cumulative reward: 8.030
Cumulative reward: 7.930
Cumulative reward: 7.830
Cumulative reward: 7.730
Cumulative reward: 7.630
Cumulative reward: 7.530
Cumulative reward: 7.430
Cumulative reward: 10.940
Cumulative reward: 10.840
Cumulative reward: 10.740
Cumulative reward: 10.640
Cumulative reward: 10.540
Cumulative reward: 1

In [71]:
# After CMA-ES training loop finishes
best_weights = torch.tensor(es.result.xbest.reshape(action_dim, latent_dim), dtype=torch.float32).to(device)
controller.linear.weight.data = best_weights
controller.linear.bias.data.zero_()  # optional

print("Best controller weights (linear layer):")
print(controller.linear.weight.data)

print("Best controller bias (linear layer):")
print(controller.linear.bias.data)

Best controller weights (linear layer):
tensor([[ 1.3050, -0.0540, -0.2987,  ...,  0.9157, -0.4354, -0.1758],
        [ 0.1020, -0.4350, -0.4319,  ..., -0.9810, -0.5102, -0.2480],
        [ 0.1388,  0.0268,  0.1854,  ..., -0.1942, -0.1292,  1.1766]],
       device='cuda:0')
Best controller bias (linear layer):
tensor([0., 0., 0.], device='cuda:0')


In [34]:
!pip install swig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 15.4 MB/s eta 0:00:00


In [36]:
!pip install "gymnasium[box2d]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 18.0 MB/s eta 0:00:00


In [33]:
print("\nLoaded model weights:")
print(controller.linear.weight.data)


Loaded model weights:
tensor([[-0.7072,  0.8553,  0.3602,  ...,  0.0796, -0.4743, -0.8859],
        [-0.0775,  0.6017, -0.6868,  ...,  0.2533,  0.0544,  0.8735],
        [ 0.2498, -0.6557,  0.3780,  ..., -0.3160, -0.0064,  0.0145]],
       device='cuda:0')


# 1.i have changed the sequence length
# 2.changed the rollouts_steps/max_steps
# 3.random initial states while training the controller for better generalization
